# PINN 模型示例（可直接运行）本笔记演示如何根据给定的物理公式构造 PINN（Physics-Informed Neural Network）。主要步骤：1. 定义输入/输出模块（数据加载与保存）。2. 构建必要的张量变换，形成网络输入 \(x = [\|S^{	ext{pre}}\|, S^{	ext{pre}}]\)。3. 定义主网络与辅助网络，输出 \(\hat{\mathcal{L}}_1, \hat{\mathcal{L}}_2, \hat{\mathcal{L}}_3, \hat{\mathcal{L}}_4\)。4. 构建物理损失（相位与位移约束）与数值损失（L1/L2），包含 GradNorm 自适应权重。5. 使用 AdamW 预训练，再使用 L-BFGS 精调。> 若缺少真实数据，示例会生成一份小型的合成数据集。

In [ ]:
import mathimport jsonfrom pathlib import Pathfrom typing import Dict, Tuple, Optionalimport numpy as npimport pandas as pdimport torchfrom torch import nnfrom torch.utils.data import Dataset, DataLoader

## 输入/输出模块- `StressDataset`: 读取 CSV 或自动生成合成数据。- `save_predictions`: 将预测结果保存到 CSV，便于在实验管线中串联。- 输入格式：列包含 `sin_theta`, `cos_theta`, `Sxx`, `Sxy`, `Syy`。

In [ ]:
class StressDataset(Dataset):    def __init__(self, csv_path: Optional[str] = None, n_samples: int = 512, noise: float = 0.01):        if csv_path and Path(csv_path).exists():            df = pd.read_csv(csv_path)        else:            # 生成合成数据：旋转角度、主应力幅值和剪应力            theta = np.random.uniform(0, 2 * np.pi, n_samples)            s1 = np.random.uniform(0.5, 2.0, n_samples)            s2 = np.random.uniform(0.5, 2.0, n_samples)            shear = np.random.uniform(-0.5, 0.5, n_samples)            df = pd.DataFrame({                "sin_theta": np.sin(theta),                "cos_theta": np.cos(theta),                "Sxx": s1 + noise * np.random.randn(n_samples),                "Sxy": shear + noise * np.random.randn(n_samples),                "Syy": s2 + noise * np.random.randn(n_samples),            })        self.df = df    def __len__(self):        return len(self.df)    def __getitem__(self, idx):        row = self.df.iloc[idx]        spre = torch.tensor([row.Sxx, row.Sxy, row.Syy], dtype=torch.float32)        angles = torch.tensor([row.sin_theta, row.cos_theta], dtype=torch.float32)        return angles, spredef save_predictions(path: str, angles: torch.Tensor, spre: torch.Tensor, outputs: Dict[str, torch.Tensor]):    data = {        "sin_theta": angles[:, 0].cpu().numpy(),        "cos_theta": angles[:, 1].cpu().numpy(),        "Sxx": spre[:, 0].cpu().numpy(),        "Sxy": spre[:, 1].cpu().numpy(),        "Syy": spre[:, 2].cpu().numpy(),    }    for k, v in outputs.items():        data[k] = v.detach().cpu().numpy().reshape(-1)    pd.DataFrame(data).to_csv(path, index=False)    print(f"Saved predictions to {path}")

## 应变/应力变换- \(S^{	ext{pre}}\) 经过线性变换得到 \(S\) 与 \(S^{	ext{post}}\)。- 根据示意：\( x = [\|S^{	ext{pre}}\|, S^{	ext{pre}}] \)。- 旋转矩阵 \(T_1, T_2\) 以角度 \(	heta\) 生成，可根据需求替换。

In [ ]:
def build_rotation(theta: torch.Tensor) -> torch.Tensor:    '''构造 2D 旋转矩阵 T(theta)。'''    sin_t, cos_t = theta    return torch.tensor([[cos_t, -sin_t, 0.0], [sin_t, cos_t, 0.0], [0.0, 0.0, 1.0]], dtype=torch.float32)def transform_stress(angles: torch.Tensor, spre: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:    '''生成 x, S, S_post (可进一步用于物理损失)。    返回: x (输入特征), S_pre (原始), S_post (旋转后)    '''    sin_t = angles[:, 0]    cos_t = angles[:, 1]    # 构造旋转矩阵 T1 与 T2（示例：T2 = T1^T）    T1 = torch.stack([        torch.stack([cos_t, -sin_t, torch.zeros_like(sin_t)], dim=-1),        torch.stack([sin_t, cos_t, torch.zeros_like(sin_t)], dim=-1),        torch.stack([torch.zeros_like(sin_t), torch.zeros_like(sin_t), torch.ones_like(sin_t)], dim=-1),    ], dim=1)  # (B, 3, 3)    T2 = T1.transpose(1, 2)    spre_vec = spre.unsqueeze(-1)  # (B, 3, 1)    S = torch.bmm(T1, spre_vec).squeeze(-1)    S_post = torch.bmm(T2, spre_vec).squeeze(-1)    norm_spre = torch.norm(spre, dim=1, keepdim=True)    x = torch.cat([norm_spre, spre], dim=1)    return x, S, S_post

## 网络结构- 主网络输出 \(\hat{\mathcal{L}}_1, \hat{\mathcal{L}}_2, \hat{\mathcal{L}}_3, \hat{\mathcal{L}}_4\)。- 辅助网络生成物理约束项（如相位与位移修正）。- 激活：Sigmoid；优化：AdamW，然后 L-BFGS。

In [ ]:
class PINN(nn.Module):    def __init__(self, hidden: int = 64):        super().__init__()        self.feature = nn.Sequential(            nn.Linear(4, hidden),            nn.Tanh(),            nn.Linear(hidden, hidden),            nn.Tanh(),        )        self.main_head = nn.Sequential(            nn.Linear(hidden, hidden),            nn.Sigmoid(),            nn.Linear(hidden, 4),        )        self.aux_head = nn.Sequential(            nn.Linear(hidden, hidden),            nn.Sigmoid(),            nn.Linear(hidden, 2),  # phase, displacement like corrections        )    def forward(self, x: torch.Tensor) -> Dict[str, torch.Tensor]:        h = self.feature(x)        l_hat = self.main_head(h)        aux = self.aux_head(h)        return {            "L_hat": l_hat,            "aux": aux,        }

## 损失构成- 数值损失：`L_num = |L̂_1 - L_1| + |L̂_2 - L_2|`（示例使用合成标签）。- 物理损失：根据 T1/T2 变换后的相位与位移，约束 \(S^{	ext{pre}}\)、\(S\)、\(S^{	ext{post}}\)。- GradNorm 自适应权重平衡数值与物理损失，避免梯度失衡。

In [ ]:
class GradNorm:    def __init__(self, alpha: float = 1.5):        self.alpha = alpha        self.weights = nn.Parameter(torch.ones(2))  # [numerical, physical]    def compute(self, losses: torch.Tensor, base_loss: torch.Tensor) -> torch.Tensor:        grad_norms = []        for i in range(len(losses)):            g = torch.autograd.grad(losses[i], base_loss, retain_graph=True, create_graph=True, allow_unused=True)            grad_norm = torch.abs(g[0]) if g[0] is not None else torch.tensor(0.0)            grad_norms.append(grad_norm)        grad_norms = torch.stack(grad_norms)        target = grad_norms.mean() * (losses / losses.detach()) ** self.alpha        grad_loss = (self.weights * (grad_norms - target.detach())).sum()        return grad_lossdef compute_losses(model: PINN, angles: torch.Tensor, spre: torch.Tensor) -> Dict[str, torch.Tensor]:    x, S, S_post = transform_stress(angles, spre)    out = model(x)    # 合成真值（示例）：将旋转后应力的均值作为监督信号    target_L = torch.stack([S.mean(dim=1), S_post.mean(dim=1)], dim=1)    # 扩展到四个损失分量    target_L = torch.cat([target_L, target_L], dim=1)  # (B,4)    L_hat = out["L_hat"]    l1 = torch.mean(torch.abs(L_hat[:, :2] - target_L[:, :2]))    l2 = torch.mean(torch.abs(L_hat[:, 2:] - target_L[:, 2:]))    numerical = l1 + l2    # 物理损失示例：保持 S 与 S_post 的范数一致    phys_phase = torch.mean((torch.norm(S, dim=1) - torch.norm(spre, dim=1)) ** 2)    phys_disp = torch.mean((torch.norm(S_post, dim=1) - torch.norm(spre, dim=1)) ** 2)    physical = phys_phase + phys_disp    return {        "numerical": numerical,        "physical": physical,        "L_hat": L_hat,        "aux": out["aux"],        "x": x,    }

## 训练流程1. 使用 AdamW 训练若干轮，并通过 GradNorm 平衡损失。2. 切换到 L-BFGS 进行精调，提升收敛质量。

In [ ]:
def train(model: PINN, dataloader: DataLoader, epochs: int = 30, device: str = "cpu"):    model.to(device)    gn = GradNorm()    optimizer = torch.optim.AdamW(list(model.parameters()) + [gn.weights], lr=1e-3)    for epoch in range(epochs):        for angles, spre in dataloader:            angles = angles.to(device)            spre = spre.to(device)            losses = compute_losses(model, angles, spre)            base = losses["numerical"] + losses["physical"]            gn_loss = gn.compute(torch.stack([losses["numerical"], losses["physical"]]), base)            total = base + gn_loss            optimizer.zero_grad()            total.backward()            optimizer.step()        if (epoch + 1) % 10 == 0:            print(f"Epoch {epoch+1}: numerical={losses['numerical']:.4f}, physical={losses['physical']:.4f}")    # L-BFGS 精调    def closure():        optimizer_lbfgs.zero_grad()        angles, spre = next(iter(dataloader))        angles = angles.to(device)        spre = spre.to(device)        losses = compute_losses(model, angles, spre)        loss_val = losses["numerical"] + losses["physical"]        loss_val.backward()        return loss_val    optimizer_lbfgs = torch.optim.LBFGS(model.parameters(), lr=0.5, max_iter=20)    optimizer_lbfgs.step(closure)    return model

## 示例运行

In [ ]:
# 数据加载（可替换 csv_path 为真实路径）train_ds = StressDataset(csv_path=None, n_samples=256)train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)model = PINN(hidden=64)trained_model = train(model, train_loader, epochs=20)# 推理与输出angles, spre = next(iter(train_loader))with torch.no_grad():    x, _, _ = transform_stress(angles, spre)    outputs = trained_model(x)save_predictions(    "pinn_predictions.csv",    angles,    spre,    {"L1": outputs["L_hat"][:, 0], "L2": outputs["L_hat"][:, 1], "L3": outputs["L_hat"][:, 2], "L4": outputs["L_hat"][:, 3]},)